In [18]:
from fastmcp import FastMCP

mcp = FastMCP(
    name="First Server"
)

### WAHT IS TOOL - Normal python functions:

@mcp.tool
def add(a:int, b:int) -> int:
    """ THIS Can accept two numbers and return the sum of those numbers """
    return a + b

@mcp.tool
def mul(a:int, b:int = 1):
    """Can accept two numbers and return the MULTIPLICATION of those numbers """
    return a * b

#### Connect with FastMCP Client

In [19]:
from fastmcp import Client

async with Client(mcp) as client:
    print("Protocol Version", client.protocol_version)
    tools = await client.list_tools()

    print("Call the tool")
    result = await client.call_tool("add", {"a":14, "b": 3} )
    print(result.data)



Protocol Version 2026-07-28
Call the tool
17


#### All about the Mcp Tools - Context / Tasks / Auth

In [20]:
from fastmcp import FastMCP, Client
import json

async def show(server:FastMCP, tool_name: str):
    async with Client(server) as client:
        tool = None
        for t in await client.list_tools():
            if t.name == tool_name:
                tool = t
                break
        print(f"Name: {tool.name}: {tool.description}" )
        print(json.dumps(tool.input_schema, indent=2))



In [21]:
await show(mcp, "mul")

Name: mul: Can accept two numbers and return the MULTIPLICATION of those numbers 
{
  "type": "object",
  "additionalProperties": false,
  "properties": {
    "a": {
      "type": "integer"
    },
    "b": {
      "default": 1,
      "type": "integer"
    }
  },
  "required": [
    "a"
  ]
}


In [26]:
from typing import Literal

@mcp.tool
def export(table:str, format: Literal["csv", "excel", "json"] = "csv"):
    """Export a table data into specific format"""
    return "Export Done"

[09/26/26 11:26:55] WARNING  Component already exists:     ]8;id=4621531;file:///Users/prateekmishra/Desktop/mcp_server_course/env/lib/python3.12/site-packages/fastmcp/server/providers/local_provider/local_provider.py\local_provider.py]8;;\:]8;id=4621532;file:///Users/prateekmishra/Desktop/mcp_server_course/env/lib/python3.12/site-packages/fastmcp/server/providers/local_provider/local_provider.py#192\192]8;;\
                             tool:export@                                       


In [27]:
await show(mcp, "export")

Name: export: Export a table data into specific format
{
  "type": "object",
  "additionalProperties": false,
  "properties": {
    "table": {
      "type": "string"
    },
    "format": {
      "default": "csv",
      "enum": [
        "csv",
        "excel",
        "json"
      ],
      "type": "string"
    }
  },
  "required": [
    "table"
  ]
}


#### A Pydantic model as an argument

In [30]:
from pydantic import BaseModel, Field

class Shipment(BaseModel):
    order_id:str = Field(description="This is the order id placed by the user..")
    quantity: int
    express : bool = False


@mcp.tool
def book_shipment(shipment:Shipment) -> dict:
    "Book a shipment for an order"
    return {"Booked":shipment.order_id, "qty": shipment.quantity}

[09/26/26 11:32:55] WARNING  Component already exists:     ]8;id=4621537;file:///Users/prateekmishra/Desktop/mcp_server_course/env/lib/python3.12/site-packages/fastmcp/server/providers/local_provider/local_provider.py\local_provider.py]8;;\:]8;id=4621538;file:///Users/prateekmishra/Desktop/mcp_server_course/env/lib/python3.12/site-packages/fastmcp/server/providers/local_provider/local_provider.py#192\192]8;;\
                             tool:book_shipment@                                


In [31]:
await show(mcp, "book_shipment")

Name: book_shipment: Book a shipment for an order
{
  "type": "object",
  "additionalProperties": false,
  "properties": {
    "shipment": {
      "properties": {
        "order_id": {
          "description": "This is the order id placed by the user..",
          "type": "string"
        },
        "quantity": {
          "type": "integer"
        },
        "express": {
          "default": false,
          "type": "boolean"
        }
      },
      "required": [
        "order_id",
        "quantity"
      ],
      "type": "object"
    }
  },
  "required": [
    "shipment"
  ]
}


#### Exception / Error Handeling

In [37]:
from fastmcp.exceptions import ToolError

@mcp.tool
def new_charges(amount:float):
    "Charge on a Card"
    if amount < 100:
        raise ToolError("Amount Value should not be less then 100 Rs.")
    return "Chanrges Done.."

In [39]:
async with Client(mcp) as client:
    res = await client.call_tool("new_charges", {"amount": 20}, raise_on_error=False)
    print(res)

[09/26/26 11:39:05] Error calling tool 'new_charges'                            


CallToolResult(content=[TextContent(type='text', text='Amount Value should not be less then 100 Rs.', annotations=None, meta=None)], structured_content=None, meta={'io.modelcontextprotocol/serverInfo': {'name': 'First Server', 'version': '4.0.10'}}, data=None, is_error=True)


#### `Context` - talking back to the client while server is running..

In [ ]:
from fastmcp import FastMCP, Context
import asyncio
ctxserver = FastMCP("cts-demo")


@ctxserver.tool
async def send_mails(emails:list[str], context:Context) -> str:
    """Send Emails to many students at a time"""

    totalStudents = len(emails)
    ### one email at a time, 2 sec to complete
    for i in range(totalStudents):
        email = emails[i]
        await context.report_progress(progress=i+1, total=totalStudents, 
                                      message=f"Sending email to : {email}")
        await asyncio.sleep(2)

    return "Email Sent to all the students"




In [42]:
async def watch_email_process(progress, totle, message):
    print(message)

async with Client(ctxserver) as client:
    res = await client.call_tool("send_mails", {"emails": ["prateek", "nikita", "ajay"]},
                                  progress_handler=watch_email_process)
    print(res)

Sending email to : prateek
Sending email to : nikita
Sending email to : ajay
CallToolResult(content=[TextContent(type='text', text='Email Sent to all the students', annotations=None, meta=None)], structured_content={'result': 'Email Sent to all the students'}, meta={'fastmcp': {'wrap_result': True}, 'io.modelcontextprotocol/serverInfo': {'name': 'cts-demo', 'version': '4.0.10'}}, data='Email Sent to all the students', is_error=False)


##### Long Runing Task

In [ ]:
import fastmcp_tasks
from fastmcp import FastMCP, Client
from fastmcp.dependencies import Progress
from fastmcp.utilities.tasks import TaskConfig
from fastmcp_tasks import TasksExtension, call_tool_task

mcp = FastMCP("Task Server")
mcp.add_extension(TasksExtension())

@mcp.tool(task=TaskConfig(mode="required"))
async def slow_computation(email_counts:int, progress: Progress = Progress()) -> str:
    "A long running task / opearation.."
    await progress.set_total(email_counts)
    for i in range(email_counts):
        ### sending the email code 
        await asyncio.sleep(1)
        await progress.set_message(f"Sending Email {i}/{email_counts}")
        await progress.increment()
    return f"Completed in {email_counts} seconds.."

In [49]:
async with Client(mcp) as client:
    task = await call_tool_task(client, "slow_computation", {"email_counts": 10})
    print("Task Id: ", task.task_id)

    while True:
        status = await task.status()
        print("Status: ", status.status, status.status_message)

        if status.status == "completed":
            break

        await asyncio.sleep(3)

Task Id:  so2DvKisuKsyCJTDk5xZSU_De5rjfu3y6Ldx1Ed5tbk
Status:  working None
Status:  working Sending Email 1/10
Status:  working Sending Email 4/10
Status:  working Sending Email 7/10
Status:  completed None


#### JWT Authentication for MCP Server

In [50]:
#### Server Code in auth_macp_server.py

In [51]:
TOKEN_GENERATOR_URL = "http://127.0.0.1:8001"
MCP_SERVER_URL = "http://127.0.0.1:8000/mcp"

In [75]:
from fastmcp import Client
import httpx

async def getToken():
    async with httpx.AsyncClient() as http:
        res = await http.post(
            f"{TOKEN_GENERATOR_URL}/token",
            json={"user_id": "Ajay", "email":"ajay@g.com", "password": "12334"}
        )
        if res.status_code != 200:
            return "No Token Found"

        data = res.json()
        token = data["access_token"]
        return token
    

In [76]:
token = await getToken()

In [77]:
async with Client(MCP_SERVER_URL, auth=token) as client:
    tools = await client.list_tools()
    result = await client.call_tool("whoami", {})
    print(result.data)

{'client_id': 'Ajay', 'email': 'ajay@g.com'}


##### Google OAuth Authentication

In [78]:
GOOGLE_CLIENT_ID = "208659134695-es62qbtpcmau8i18itfr11qn6rldrsp4.apps.googleusercontent.com"
GOOGLE_CLIEN_SECRET = "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"
MCP_SERVER_URL = "http://localhost:8000"

In [3]:
from fastmcp import Client
import asyncio

async def main():
    # The client will automatically handle Google OAuth
    async with Client("http://localhost:8000/mcp", auth="oauth") as client:
        # First-time connection will open Google login in your browser
        print("✓ Authenticated with Google!")
        
        # Test the protected tool
        result = await client.call_tool("get_user_info")
        user_info = result.data
        print(f"Google user: {user_info['email']}")
        print(f"Name: {user_info['name']}")

[09/26/26 16:00:53] INFO     OAuth authorization URL:                                                  ]8;id=2830308;file:///Users/prateekmishra/Desktop/mcp_server_course/env/lib/python3.12/site-packages/fastmcp/client/auth/oauth.py\oauth.py]8;;\:]8;id=2830309;file:///Users/prateekmishra/Desktop/mcp_server_course/env/lib/python3.12/site-packages/fastmcp/client/auth/oauth.py#399\399]8;;\
                             http://localhost:8000/authorize?response_type=code&client_id=02a36698-69b             
                             c-40e5-9e53-6ebbf7f841e7&redirect_uri=http%3A%2F%2Flocalhost%3A62538%2Fca             
                             llback&state=tDpEG42jSMjCtvc1cxwmtvA6UWggzRujdaIh2DJqjH0&code_challenge=L             
                             mU1uTRPjhAn665siOIiUHKw5sXlfMl3U8LKrOTJMjg&code_challenge_method=S256&res             
                             ource=http%3A%2F%2Flocalhost%3A8000%2Fmcp&scope=openid+https%3A%2F%2Fwww.             
                             googleapis.com%2Fauth%2Fuserinfo.email                                                

                    INFO     🎧 OAuth callback server started on http://localhost:62538                ]8;id=2830314;file:///Users/prateekmishra/Desktop/mcp_server_course/env/lib/python3.12/site-packages/fastmcp/client/auth/oauth.py\oauth.py]8;;\:]8;id=2830315;file:///Users/prateekmishra/Desktop/mcp_server_course/env/lib/python3.12/site-packages/fastmcp/client/auth/oauth.py#420\420]8;;\

OAuth flow error
  + Exception Group Traceback (most recent call last):
  |   File "/Users/prateekmishra/Desktop/mcp_server_course/env/lib/python3.12/site-packages/mcp/client/auth/oauth2.py", line 785, in _auth_flow
  |     token_response = yield await self._perform_authorization()
  |                            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "/Users/prateekmishra/Desktop/mcp_server_course/env/lib/python3.12/site-packages/fastmcp/client/auth/oauth.py", line 379, in _perform_authorization
  |     return await super()._perform_authorization()
  |            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "/Users/prateekmishra/Desktop/mcp_server_course/env/lib/python3.12/site-packages/mcp/client/auth/oauth2.py", line 380, in _perform_authorization
  |     auth_code, code_verifier = await self._perform_authorization_code_grant()
  |                                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "/Users/prateekmishra/Desktop/mcp_server_course/env/

RuntimeError: Client failed to connect: Access was denied by the authorization server.